## 04 — Visualisation Exploration

Exploratory notebook working through chart options for each of the five poster stories. Purpose is to identify the best chart type, scale, and layout before building the final Processing sketches.

Each story section tries 2–3 chart approaches and prints observations to guide the final design decision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

OUT_TABLES = Path.cwd().parent / "outputs" / "tables"
OUT_FIGURES = Path.cwd().parent / "outputs" / "figures"
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

# Load all story CSVs
s1 = pd.read_csv(OUT_TABLES / "story1_uk_posture_trends.csv")
s2 = pd.read_csv(OUT_TABLES / "story2_board_priority_by_size.csv")
s3 = pd.read_csv(OUT_TABLES / "story3_pathway_analysis.csv")
s4 = pd.read_csv(OUT_TABLES / "story4_control_adoption.csv")
s5 = pd.read_csv(OUT_TABLES / "story5_governance_vs_breach_size.csv")

# Shared style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

YEARS = list(range(2018, 2026))
SIZE_ORDER = ["Micro", "Small", "Medium", "Large"]
SIZE_COLOURS = {"Micro": "#4e9af1", "Small": "#f4a235", "Medium": "#2ebd85", "Large": "#e05c5c"}

print("Loaded all story CSVs.")

---
## Story 1 — UK Posture Over 8 Years

Variables: `governance_score_A`, `governance_score_B`, `breach_pct`, `participant_count`

Key question: does governance track alongside breach prevalence? Do Score A and B tell different stories?

In [ ]:
# --- S1 Option A: Dual-axis line chart (governance left, breach right) ---
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(s1["year"], s1["governance_score_A"], marker="o", color="#2ebd85", label="Governance Score A")
ax1.plot(s1["year"], s1["governance_score_B"], marker="s", color="#2ebd85", linestyle="--", label="Governance Score B")
ax2.plot(s1["year"], s1["breach_pct"],         marker="^", color="#e05c5c", label="Breach %")

ax1.set_ylabel("Governance Score (0–1)", color="#2ebd85")
ax2.set_ylabel("Breach Prevalence", color="#e05c5c")
ax1.set_xlabel("Year")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.set_xticks(YEARS)
ax1.set_ylim(0, 0.8)
ax2.set_ylim(0, 0.8)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
plt.title("Story 1A — Dual-axis: Governance vs Breach")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s1_option_a_dual_axis.png")
plt.show()

In [ ]:
# --- S1 Option B: Score A vs Score B comparison (same axis) ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

# Left: both governance scores
axes[0].plot(s1["year"], s1["governance_score_A"], marker="o", label="Score A (binary sum)")
axes[0].plot(s1["year"], s1["governance_score_B"], marker="s", linestyle="--", label="Score B (3-component)")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_xticks(YEARS); axes[0].set_xticklabels(YEARS, rotation=45)
axes[0].set_title("Governance composites: A vs B")
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 0.7)

# Right: breach with participant count as bar background
ax_bar = axes[1].twinx()
ax_bar.bar(s1["year"], s1["participant_count"], color="#cccccc", alpha=0.4, label="n respondents")
ax_bar.set_ylabel("Respondents (n)", color="grey")
axes[1].plot(s1["year"], s1["breach_pct"], marker="^", color="#e05c5c", label="Breach %")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_xticks(YEARS); axes[1].set_xticklabels(YEARS, rotation=45)
axes[1].set_title("Breach prevalence + sample size")
axes[1].legend(fontsize=8, loc="upper left")
axes[1].set_ylim(0, 0.7)

plt.suptitle("Story 1B — Split panels")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s1_option_b_split.png")
plt.show()

In [ ]:
# --- S1 Option C: Stacked area showing gap between governance and breach ---
fig, ax = plt.subplots(figsize=(9, 4))

ax.fill_between(s1["year"], s1["governance_score_B"], s1["breach_pct"],
                where=s1["breach_pct"] >= s1["governance_score_B"],
                alpha=0.2, color="#e05c5c", label="Breach > Governance gap")
ax.fill_between(s1["year"], s1["governance_score_B"], s1["breach_pct"],
                where=s1["breach_pct"] < s1["governance_score_B"],
                alpha=0.2, color="#2ebd85", label="Governance > Breach gap")
ax.plot(s1["year"], s1["governance_score_B"], marker="o", color="#2ebd85", label="Governance Score B")
ax.plot(s1["year"], s1["breach_pct"],         marker="^", color="#e05c5c", label="Breach %")

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xticks(YEARS)
ax.set_ylim(0, 0.7)
ax.set_xlabel("Year")
ax.legend(fontsize=8)
plt.title("Story 1C — Gap chart: Governance vs Breach")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s1_option_c_gap.png")
plt.show()

print("\nObservation: governance rises broadly 2018–2022, dips 2023, recovers 2025.")
print("Breach has no clear downward trend despite rising governance — important finding.")
print("Score B consistently ~0.07–0.09 higher than Score A (policy coverage dominates A less).")

---
## Story 2 — Board Priority by Org Size

Variables: `year`, `org_size`, `board_priority_pct`, `board_priority_mean_norm`

Key question: do larger orgs consistently prioritise cyber more? Has the gap between sizes narrowed?

In [ ]:
# --- S2 Option A: Line chart per org size over time ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for size in SIZE_ORDER:
    d = s2[s2["org_size"] == size]
    axes[0].plot(d["year"], d["board_priority_pct"], marker="o",
                 color=SIZE_COLOURS[size], label=size)
    axes[1].plot(d["year"], d["board_priority_mean_norm"], marker="s",
                 color=SIZE_COLOURS[size], linestyle="--", label=size)

for ax, title in zip(axes, ["Binary: % High/Very High Priority", "Continuous: Normalised Mean Priority"]):
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_xticks(YEARS); ax.set_xticklabels(YEARS, rotation=45)
    ax.set_title(title)
    ax.set_ylim(0.5, 1.05)
    ax.legend(fontsize=8)

plt.suptitle("Story 2A — Board Priority by Size: Binary vs Continuous")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s2_option_a_lines.png")
plt.show()

In [ ]:
# --- S2 Option B: Slope chart — 2018 vs 2025 directional change by size ---
fig, ax = plt.subplots(figsize=(6, 5))

start = s2[s2["year"] == 2018].set_index("org_size")["board_priority_pct"]
end   = s2[s2["year"] == 2025].set_index("org_size")["board_priority_pct"]

for size in SIZE_ORDER:
    colour = SIZE_COLOURS[size]
    ax.plot([0, 1], [start[size], end[size]], marker="o", color=colour, lw=2, label=size)
    ax.text(-0.05, start[size], f"{start[size]:.0%}", ha="right", va="center", color=colour, fontsize=9)
    ax.text(1.05,  end[size],   f"{end[size]:.0%}",   ha="left",  va="center", color=colour, fontsize=9)

ax.set_xlim(-0.3, 1.3)
ax.set_xticks([0, 1]); ax.set_xticklabels(["2018", "2025"], fontsize=11)
ax.set_yticks([]); ax.spines["left"].set_visible(False); ax.spines["bottom"].set_visible(False)
ax.legend(loc="center right", fontsize=8)
plt.title("Story 2B — Slope Chart: Priority 2018 → 2025")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s2_option_b_slope.png")
plt.show()

In [ ]:

# --- S2 Option C: Heatmap — year × org size ---
pivot = s2.pivot(index="org_size", columns="year", values="board_priority_pct")
pivot = pivot.loc[SIZE_ORDER]  # enforce size order

fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(pivot, annot=True, fmt=".0%", cmap="YlGn", ax=ax,
            vmin=0.6, vmax=1.0, linewidths=0.5,
            cbar_kws={"format": mticker.PercentFormatter(xmax=1)})
ax.set_xlabel("Year"); ax.set_ylabel("Org Size")
plt.title("Story 2C — Heatmap: Board Priority % by Size × Year")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s2_option_c_heatmap.png")
plt.show()

print("\nObservation: consistent size gradient (Large > Medium > Small > Micro) every year.")
print("Micro shows most volatility. All sizes dip slightly in 2023.")
print("The heatmap shows this pattern most clearly at a glance.")


---
## Story 3 — Pathway Analysis: Policy → Management → Breach

Variables: `has_policy`, `actively_managed`, `breached`, `proportion`

Key question: does having a policy + active management reduce breach rates? What proportion of orgs are in each pathway stage?

In [ ]:
# --- S3 Option A: Grouped bar chart of all 8 combinations ---
s3["label"] = s3.apply(
    lambda r: f"{'Policy' if r.has_policy else 'No Policy'} / "
              f"{'Managed' if r.actively_managed else 'Not Managed'} / "
              f"{'Breached' if r.breached else 'Not Breached'}",
    axis=1
)
colours = ["#e05c5c" if r.breached else "#2ebd85" for _, r in s3.iterrows()]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(s3["label"], s3["proportion"], color=colours)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlabel("Proportion of all organisations (weighted, all years pooled)")
for bar, val in zip(bars, s3["proportion"]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f"{val:.1%}", va="center", fontsize=8)
plt.title("Story 3A — Pathway combinations (all years pooled)")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s3_option_a_grouped_bar.png")
plt.show()

In [ ]:
# --- S3 Option B: Conditional breach rates at each stage ---
# For each policy/management combination, what % got breached?
breach_rates = []
for hp in [0, 1]:
    for am in [0, 1]:
        subset = s3[(s3["has_policy"] == hp) & (s3["actively_managed"] == am)]
        total = subset["proportion"].sum()
        breached = subset[subset["breached"] == 1]["proportion"].sum()
        breach_rates.append({
            "group": f"{'Policy' if hp else 'No Policy'} +\n{'Managed' if am else 'Not Managed'}",
            "breach_rate": breached / total if total > 0 else np.nan,
            "total_share": total,
        })

br_df = pd.DataFrame(breach_rates)
print(br_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colours_br = ["#4e9af1" if "Policy" in g and "Managed" in g and "No" not in g
              else "#aaaaaa" for g in br_df["group"]]
axes[0].bar(br_df["group"], br_df["breach_rate"], color="#e05c5c", alpha=0.8)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_title("Conditional breach rate by governance stage")
axes[0].set_ylim(0, 0.8)
for i, (_, row) in enumerate(br_df.iterrows()):
    axes[0].text(i, row["breach_rate"] + 0.01, f"{row['breach_rate']:.0%}", ha="center", fontsize=9)

axes[1].bar(br_df["group"], br_df["total_share"], color="#4e9af1", alpha=0.8)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_title("Share of total population in each stage")
for i, (_, row) in enumerate(br_df.iterrows()):
    axes[1].text(i, row["total_share"] + 0.003, f"{row['total_share']:.0%}", ha="center", fontsize=9)

plt.suptitle("Story 3B — Conditional breach rates by governance stage")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s3_option_b_conditional.png")
plt.show()

In [ ]:
# --- S3 Option C: Stacked bar showing breached/not-breached split per stage ---
br_df_breach  = s3[s3["breached"] == 1].copy()
br_df_nobreach = s3[s3["breached"] == 0].copy()

labels = br_df["group"].values
not_br = br_df_nobreach.sort_values(["has_policy","actively_managed"])["proportion"].values
br_v   = br_df_breach.sort_values(["has_policy","actively_managed"])["proportion"].values

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, not_br, label="Not Breached", color="#2ebd85", alpha=0.8)
ax.bar(labels, br_v, bottom=not_br, label="Breached", color="#e05c5c", alpha=0.8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("Proportion of total population")
ax.legend()
plt.title("Story 3C — Population share per governance stage, breached vs not")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s3_option_c_stacked.png")
plt.show()

print("\nKey finding: 'No Policy / Not Managed' is the largest group (~48% of population).")
print("Breach rate rises from No Policy+Not Managed → Policy+Managed — possibly because")
print("larger/more complex orgs both invest in governance AND face more attacks.")

---
## Story 4 — Control Adoption Over Time

Variables: `year`, `ce_adoption_pct`, `ten_steps_pct`, `breach_nonadopter_pct`

Key question: is CE adoption falling? Does being a non-adopter increasingly predict a breach?

In [ ]:
# --- S4 Option A: Line chart with three series ---
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(s4["year"], s4["ce_adoption_pct"],   marker="o", color="#4e9af1", label="Cyber Essentials adoption")
ax1.plot(s4["year"], s4["ten_steps_pct"],     marker="s", color="#f4a235", label="Any 10 Steps adoption")
ax2.plot(s4["year"], s4["breach_nonadopter_pct"], marker="^", color="#e05c5c",
         linestyle="--", label="Breach rate (non-adopters)")

ax1.set_ylabel("Adoption %", color="#333")
ax2.set_ylabel("Breach rate — non-adopters", color="#e05c5c")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.set_xticks(YEARS)
ax1.set_ylim(0, 1.1)
ax2.set_ylim(0, 0.4)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
plt.title("Story 4A — Control adoption trends with non-adopter breach rate")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s4_option_a_lines.png")
plt.show()

In [ ]:
# --- S4 Option B: Stacked area (CE + non-CE) with breach overlay ---
fig, ax = plt.subplots(figsize=(9, 4))

ax.stackplot(s4["year"],
             s4["ce_adoption_pct"],
             1 - s4["ce_adoption_pct"],
             labels=["CE certified", "Not CE certified"],
             colors=["#4e9af1", "#dddddd"], alpha=0.7)

ax2 = ax.twinx()
ax2.plot(s4["year"], s4["breach_nonadopter_pct"], marker="^", color="#e05c5c",
         lw=2, label="Breach — non-adopters")
ax2.set_ylabel("Breach rate (non-adopters)", color="#e05c5c")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.set_ylim(0, 0.4)

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("Share of organisations")
ax.set_xticks(YEARS)
ax.set_ylim(0, 1)
ax.legend(loc="lower left", fontsize=8)
ax2.legend(loc="upper right", fontsize=8)
plt.title("Story 4B — CE adoption share + non-adopter breach risk")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s4_option_b_stacked.png")
plt.show()

print("\nObservation: CE adoption has fallen significantly from ~53% (2020) to ~23% (2025).")
print("10 Steps adoption is near-universal (88–96%) — may be due to low bar (any 1 of 10).")
print("Non-adopter breach rate is low in 2018 (~5%) — possibly very small sample; rises after.")
print("Consider flagging 2018 non-adopter figure as unreliable (tiny n).")

---
## Story 5 — Governance Score vs Breach by Org Size

Variables: `year`, `org_size`, `governance_score_A/B`, `breach_pct`, `org_count`

Key question: does higher governance predict lower breach rates? Does this relationship hold consistently across sizes?

In [ ]:
# --- S5 Option A: Dual line chart (governance + breach) per size, small multiples ---
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True)

for i, size in enumerate(SIZE_ORDER):
    d = s5[s5["org_size"] == size]
    colour = SIZE_COLOURS[size]

    ax_gov = axes[0, i]
    ax_br  = axes[1, i]

    ax_gov.plot(d["year"], d["governance_score_A"], marker="o", color=colour, lw=1.5, label="Score A")
    ax_gov.plot(d["year"], d["governance_score_B"], marker="s", color=colour, lw=1.5,
                linestyle="--", alpha=0.6, label="Score B")
    ax_gov.set_title(size, fontsize=10, color=colour)
    ax_gov.set_ylim(0, 1)
    ax_gov.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    if i == 0: ax_gov.set_ylabel("Governance Score")

    ax_br.plot(d["year"], d["breach_pct"], marker="^", color="#e05c5c", lw=1.5)
    ax_br.set_ylim(0, 1)
    ax_br.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax_br.set_xticks(YEARS); ax_br.set_xticklabels(YEARS, rotation=90, fontsize=7)
    if i == 0: ax_br.set_ylabel("Breach %")

axes[0, 0].legend(fontsize=7)
plt.suptitle("Story 5A — Governance and Breach by org size (small multiples)")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s5_option_a_multiples.png")
plt.show()

In [ ]:
# --- S5 Option B: Scatter — governance_score_B vs breach_pct, coloured by size ---
# Each point = one (year × size) observation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, score_col, title in zip(
    axes,
    ["governance_score_A", "governance_score_B"],
    ["Score A vs Breach", "Score B vs Breach"]
):
    for size in SIZE_ORDER:
        d = s5[s5["org_size"] == size]
        ax.scatter(d[score_col], d["breach_pct"],
                   color=SIZE_COLOURS[size], s=d["org_count"]/8,
                   label=size, alpha=0.7, edgecolors="white", lw=0.5)
        # Connect years with a line
        ax.plot(d[score_col], d["breach_pct"], color=SIZE_COLOURS[size], alpha=0.3, lw=1)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_xlabel(f"Governance ({score_col})")
    ax.set_ylabel("Breach %")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle("Story 5B — Scatter: Governance vs Breach (bubble size = n respondents)")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "s5_option_b_scatter.png")
plt.show()

print("\nObservation: larger orgs have higher governance scores AND higher breach rates.")
print("This confirms exposure bias: larger orgs face more attacks, not that governance fails.")
print("Within each size band, the governance-breach relationship is less clear year-to-year.")

In [ ]:
# --- S5 Option C: Correlation within size bands (governance vs breach across years) ---
from scipy.stats import pearsonr

print("Pearson correlation: governance_score_B vs breach_pct within each size band")
print(f"{'Size':<10} {'r (Score A)':<15} {'r (Score B)':<15} {'n years'}")
for size in SIZE_ORDER:
    d = s5[s5["org_size"] == size].dropna(subset=["governance_score_A", "governance_score_B", "breach_pct"])
    rA, pA = pearsonr(d["governance_score_A"], d["breach_pct"])
    rB, pB = pearsonr(d["governance_score_B"], d["breach_pct"])
    print(f"{size:<10} {rA:+.3f} (p={pA:.2f})  {rB:+.3f} (p={pB:.2f})  {len(d)}")

print("\nNote: n=8 years per size band means these correlations have very low statistical power.")
print("Positive r means higher governance → higher breach in this size band (exposure confound).")

---
## Summary: Recommended chart directions

| Story | Recommended | Reason |
|---|---|---|
| 1 | Option C (gap chart) | Shows the governance–breach relationship directly; most visually impactful |
| 2 | Option C (heatmap) or A (lines) | Heatmap is most readable for the size × year grid; lines better for trend narrative |
| 3 | Option B (conditional rates) | Bar chart of conditional breach rates tells the clearest causal story |
| 4 | Option A (dual-axis lines) | All three series visible; CE decline and breach risk readable together |
| 5 | Option A (small multiples) | Most honest — shows within-size trends without cross-size confounding |

Revise these notes as you review the saved figures in `outputs/figures/`.